In [9]:
import kagglehub
import os
from kagglehub import KaggleDatasetAdapter

dataset_id = "kundanbedmutha/instagram-analytics-dataset"

# Download dataset
path = kagglehub.dataset_download(dataset_id)

print("Dataset path:", path)

# Auto-detect CSV file
csv_file = None
for file in os.listdir(path):
    if file.endswith(".csv"):
        csv_file = file
        break

if not csv_file:
    raise Exception("No CSV file found in dataset")

print("Using file:", csv_file)

# Load dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    dataset_id,
    csv_file
)

print(df.head())

Dataset path: C:\Users\haiwa\.cache\kagglehub\datasets\kundanbedmutha\instagram-analytics-dataset\versions\3
Using file: Instagram_Analytics.csv


C:\Users\haiwa\AppData\Local\Temp\ipykernel_18092\707307572.py:25: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


     post_id  account_id account_type  follower_count media_type  \
0  IG0000001           7        brand            3551       reel   
1  IG0000002          20      creator           31095      image   
2  IG0000003          15        brand            8167       reel   
3  IG0000004          11      creator            9044   carousel   
4  IG0000005           8      creator           15986       reel   

  content_category traffic_source  has_call_to_action        post_datetime  \
0       Technology      Home Feed                   1  2024-11-30 06:00:00   
1          Fitness       Hashtags                   1  2025-08-15 15:00:00   
2           Beauty     Reels Feed                   0  2025-09-11 16:00:00   
3            Music       External                   0  2025-09-18 03:00:00   
4       Technology        Profile                   0  2025-03-21 09:00:00   

    post_date  ...  comments shares  saves  reach  impressions  \
0  2024-11-30  ...         5      7     34   4327       

In [10]:
df


,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,1,2024-11-30 06:00:00,2024-11-30,...,5,7,34,4327,6230,0.0385,899,100,7,medium
1,IG0000002,20,creator,31095,image,Fitness,Hashtags,1,2025-08-15 15:00:00,2025-08-15,...,10,21,68,7451,8268,0.0663,805,122,5,viral
2,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,0,2025-09-11 16:00:00,2025-09-11,...,2,1,22,1639,2616,0.0531,758,115,8,high
3,IG0000004,11,creator,9044,carousel,Music,External,0,2025-09-18 03:00:00,2025-09-18,...,0,7,0,2877,3171,0.0309,402,115,7,medium
4,IG0000005,8,creator,15986,reel,Technology,Profile,0,2025-03-21 09:00:00,2025-03-21,...,8,5,21,5350,8503,0.0221,155,112,9,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29994,IG0029995,5,brand,10739,carousel,Travel,Reels Feed,0,2024-12-18 10:00:00,2024-12-18,...,1,2,5,1564,2493,0.0032,124,127,8,low
29995,IG0029996,3,brand,10018,image,Beauty,Hashtags,0,2025-05-05 15:00:00,2025-05-05,...,2,1,7,2042,2492,0.0209,310,114,12,low
29996,IG0029997,18,creator,7486,image,Photography,Explore,1,2025-05-26 10:00:00,2025-05-26,...,10,16,59,5887,7528,0.0558,223,115,4,high
29997,IG0029998,6,creator,10034,carousel,Technology,Explore,1,2025-08-02 19:00:00,2025-08-02,...,3,0,19,5372,6312,0.0333,978,124,4,medium


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ---------------- CLEAN COLUMN NAMES ---------------- #
df.columns = df.columns.str.strip().str.lower()

# ---------------- HANDLE DATETIME ---------------- #
df["post_datetime"] = pd.to_datetime(df["post_datetime"], errors='coerce')

# extract useful features
df["hour"] = df["post_datetime"].dt.hour
df["day"] = df["post_datetime"].dt.day
df["month"] = df["post_datetime"].dt.month

# ---------------- DROP USELESS COLUMNS ---------------- #
drop_cols = [
    "post_id", "account_id", "post_datetime", "post_date",
    "performance_bucket_label"  # label column (not needed for regression)
]

df = df.drop(columns=drop_cols, errors='ignore')

# ---------------- HANDLE NULLS ---------------- #
df = df.dropna()

# ---------------- DEFINE TARGET ---------------- #
y = df["engagement_rate"]

# ---------------- DEFINE FEATURES ---------------- #
X = df.drop(columns=["engagement_rate"])

# ---------------- SPLIT TYPES ---------------- #
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "bool"]).columns

# ---------------- PREPROCESSOR ---------------- #
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# ---------------- MODEL PIPELINE ---------------- #
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

# ---------------- TRAIN TEST SPLIT ---------------- #
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------- TRAIN ---------------- #
model.fit(X_train, y_train)

print("Model trained successfully 🚀")

# ---------------- EVALUATE ---------------- #
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"R2 Score: {r2}")

# ---------------- SAMPLE PREDICTION ---------------- #
sample = X.iloc[[0]]
prediction = model.predict(sample)

print("Sample Prediction:", prediction[0])

Model trained successfully 🚀
MAE: 0.005642800333333332
R2 Score: 0.6492047724663659
Sample Prediction: 0.03855599999999998
